# Differential Equations — Session 5
## Section 2.2: Separable Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. Recognize a separable first-order equation.
2. Separate variables and integrate correctly.
3. Apply initial conditions to implicit or explicit solutions.
4. identify constant solutions that may be lost during division.
5. determine an appropriate interval of definition.
6. interpret implicit solution curves.
7. use a definite integral when no elementary antiderivative exists.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–15 min | Recognition and method |
| 15–35 min | Explicit separable example |
| 35–52 min | Implicit solution curves and intervals |
| 52–65 min | Lost equilibrium solutions |
| 65–78 min | Logistic example |
| 78–87 min | Integral-defined solutions |
| 87–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.2-A — Separable equation

A first-order equation is **separable** if it can be written as

$$
\frac{dy}{dx}=g(x)h(y).
$$

On a region where $h(y)\ne0$, the variables can be separated:

$$
\frac{1}{h(y)}\,dy=g(x)\,dx.
$$

### Theorem 2.2-B — Implicit solution formula

Assume $g$ and $1/h$ are continuous on the intervals under consideration. If

$$
H'(y)=\frac{1}{h(y)}
\qquad\text{and}\qquad
G'(x)=g(x),
$$

then every nonconstant solution satisfies

$$
H(y)=G(x)+C.
$$

Conversely, any differentiable branch of this relation that stays where $h(y)\ne0$ is a solution.

### Proposition 2.2-C — Equilibrium solutions

Every root $r$ of $h(y)$ produces a constant solution

$$
y(x)=r.
$$

These solutions may disappear algebraically when the equation is divided by $h(y)$, so they must be recorded before separation.

### Interval principle

After solving, the maximal interval must avoid:

- points where the differential equation is undefined,
- points where the explicit branch is not differentiable,
- and finite points where the solution becomes unbounded.

### Classroom Checkpoint — Do Not Lose an Equilibrium

When separating

$$
y'=g(x)h(y),
$$

why must solutions satisfying $h(y)=0$ be checked before dividing by $h(y)$?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Method

A separable equation has the form

$$
\frac{dy}{dx}=g(x)h(y).
$$

Where $h(y)\ne0$, rewrite it as

$$
\frac{1}{h(y)}\,dy=g(x)\,dx
$$

and integrate:

$$
\int \frac{1}{h(y)}\,dy=\int g(x)\,dx+C.
$$

Before dividing by $h(y)$, record all roots of $h(y)$ because they may produce constant solutions.

## 2. Explicit example with finite-time blow-up

Solve

$$
y'=x(1+y^2),\qquad y(0)=0.
$$

Separate and integrate:

$$
\frac{dy}{1+y^2}=x\,dx,
$$

$$
\arctan y=\frac{x^2}{2}+C.
$$

The initial condition gives $C=0$, so

$$
y=\tan\left(\frac{x^2}{2}\right).
$$

In [ ]:
x = np.linspace(-1.7, 1.7, 700)
y = np.tan(x**2/2)

plt.plot(x, y, linewidth=2)
plt.axvline(-np.sqrt(np.pi), linestyle="--")
plt.axvline(np.sqrt(np.pi), linestyle="--",
            label=r"nearest singular points $x=\pm\sqrt{\pi}$")
plt.ylim(-10, 10)
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"$y' = x(1+y^2),\ y(0)=0$")
plt.legend()
plt.show()

The maximal interval containing $x=0$ is

$$
-\sqrt{\pi}<x<\sqrt{\pi}.
$$

The differential equation is smooth everywhere, but this particular solution becomes unbounded in finite $x$.

## 3. An implicit solution curve

Solve

$$
y\frac{dy}{dx}=-2x,\qquad y(1)=-2.
$$

Then

$$
y\,dy=-2x\,dx,
$$

$$
\frac{y^2}{2}=-x^2+C,
$$

or

$$
2x^2+y^2=C_1.
$$

The initial point gives $C_1=6$. The implicit curve is an ellipse, but the IVP selects the lower branch

$$
y=-\sqrt{6-2x^2}.
$$

In [ ]:
x = np.linspace(-np.sqrt(3), np.sqrt(3), 600)
upper = np.sqrt(np.maximum(6-2*x**2, 0))
lower = -upper

plt.plot(x, upper, label="upper branch")
plt.plot(x, lower, linewidth=3, label="IVP solution branch")
plt.scatter([1], [-2], s=80)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Implicit family member $2x^2+y^2=6$")
plt.legend()
plt.show()

The branch is differentiable only for

$$
-\sqrt{3}<x<\sqrt{3}.
$$

## 4. Losing solutions by division

Consider

$$
y'=x(y-2)(y+1).
$$

The constant functions

$$
y=2,\qquad y=-1
$$

are solutions.

If we divide by $(y-2)(y+1)$, those values are excluded from the separated equation. Always list equilibrium solutions before division.

In [ ]:
def lost_rhs(x, y):
    return x*(y-2)*(y+1)

slope_field(lost_rhs, xlim=(-2.5, 2.5), ylim=(-2.5, 3.5), density=23,
            title=r"$y'=x(y-2)(y+1)$")

plt.axhline(-1, linestyle="--", label=r"$y=-1$")
plt.axhline(2, linestyle="--", label=r"$y=2$")

for y0 in [-2, 0, 1, 2.8]:
    sol = solve_ivp(lambda x, y: lost_rhs(x, y[0]), (0, 2), [y0],
                    t_eval=np.linspace(0, 2, 400))
    plt.plot(sol.t, sol.y[0], linewidth=2)

plt.legend()
plt.show()

## 5. Logistic growth by separation

For

$$
P'=rP\left(1-\frac{P}{K}\right),
$$

the constant solutions are $P=0$ and $P=K$. For nonconstant solutions,

$$
\frac{dP}{P(1-P/K)}=r\,dt.
$$

Partial fractions produce the logistic family

$$
P(t)=\frac{K}{1+Ae^{-rt}}.
$$

In [ ]:
def logistic_demo(P0=80, r=0.5, K=500):
    t = np.linspace(0, 20, 600)
    A = (K-P0)/P0
    P = K/(1+A*np.exp(-r*t))
    plt.plot(t, P, linewidth=2, label="solution")
    plt.axhline(K, linestyle="--", label="carrying capacity")
    plt.scatter([0], [P0], s=70)
    plt.xlabel("t")
    plt.ylabel("P(t)")
    plt.title("Logistic solution obtained by separation")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        logistic_demo,
        P0=FloatSlider(min=10, max=900, step=10, value=80),
        r=FloatSlider(min=0.1, max=1.2, step=0.1, value=0.5),
        K=FloatSlider(min=100, max=1000, step=50, value=500)
    )
else:
    logistic_demo()

## 6. Integral-defined solutions

Not every continuous function has an elementary antiderivative.

For

$$
y'=e^{-x^4},\qquad y(0)=1,
$$

a correct exact solution is

$$
y(x)=1+\int_0^x e^{-t^4}\,dt.
$$

In [ ]:
x_vals = np.linspace(-2, 2, 300)
y_vals = np.array([
    1 + quad(lambda t: np.exp(-t**4), 0, x)[0]
    for x in x_vals
])

plt.plot(x_vals, y_vals, linewidth=2)
plt.scatter([0], [1], s=70)
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"$y=1+\int_0^x e^{-t^4}\,dt$")
plt.show()

## Interactive exploration — Blow-up depends on the initial value

For

$$
y'=y^2,
\qquad
y(0)=y_0,
$$

the solution is

$$
y=\frac{y_0}{1-y_0x}.
$$

For $y_0>0$, the blow-up time is $x=1/y_0$.

In [ ]:
def separable_blowup(y0=1.0):
    if abs(y0) < 1e-10:
        x = np.linspace(-3, 3, 500)
        y = np.zeros_like(x)
        plt.plot(x, y)
        print("The equilibrium solution exists for all real x.")
    else:
        singular = 1/y0
        left = np.linspace(max(-4, singular-5), singular-0.04, 500)
        right = np.linspace(singular+0.04, min(4, singular+5), 500)
        if len(left) > 1:
            plt.plot(left, y0/(1-y0*left))
        if len(right) > 1:
            plt.plot(right, y0/(1-y0*right))
        plt.axvline(singular, linestyle="--", label=fr"$x={singular:.3f}$")
        plt.ylim(-10, 10)
        plt.legend()
        print("Singular point:", singular)
    plt.scatter([0], [y0], s=70)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(r"$y'=y^2$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        separable_blowup,
        y0=FloatSlider(min=-2, max=2, step=0.1, value=1)
    )
else:
    separable_blowup()

## Exit check

Solve and include all constant solutions:

$$
y'=x(y-3)^2.
$$

For nonconstant solutions,

$$
-\frac{1}{y-3}=\frac{x^2}{2}+C.
$$

Also include the equilibrium solution $y=3$.

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.